In [1]:
import sys
import os

def get_UGCE_directory():
    """Get the path of the 'UGCE-User-Guided-Counterfactual-Exploration' directory."""
    current_dir = os.getcwd()
    target_dir = 'UGCE-User-Guided-Counterfactual-Exploration'
    
    while os.path.basename(current_dir) != target_dir:
        current_dir = os.path.dirname(current_dir)
        if current_dir == os.path.dirname(current_dir):
            return None
        
    return current_dir

def get_system_slash():
    """Get the system-specific directory separator."""
    return os.sep

UGCE_dir = get_UGCE_directory()
sys.path.append(UGCE_dir)
sep = get_system_slash()
sys.path.append(UGCE_dir + get_system_slash() + 'src')

from dataLoader import *
from utils import *
from test_utils import *

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
seed_number = 42
import random

random.seed(seed_number)
np.random.seed(seed_number)

In [4]:
datasetName = "Compas"

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from aif360.sklearn.datasets import fetch_compas
import pandas as pd
import dice_ml
from dice_ml.utils import helpers

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier


dataset, target = fetch_compas()
dataset = dataset.reset_index(drop=True)
target = target.reset_index(drop=True)
TARGET_COLUMN = 'two_year_recid'
dataset[TARGET_COLUMN] = target
dataset[TARGET_COLUMN] = LabelEncoder().fit_transform(target)
dataset = dataset.drop(['c_charge_desc', 'age_cat'], axis=1)
target = dataset[TARGET_COLUMN]
datasetX = dataset.drop(['two_year_recid'], axis=1)

x_train, x_test, y_train, y_test = train_test_split(datasetX,
                                                    target,
                                                    test_size=0.2,
                                                    random_state=0,
                                                    stratify=target)

numerical = ["age", "juv_fel_count", "juv_misd_count", "juv_other_count", "priors_count"]
categorical = x_train.columns.difference(numerical)

try:
    import joblib
    model = joblib.load(f"{ugce_dir}/results/models/{datasetName}_model.pkl")
except:
    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())])

    categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    transformations = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical),
            ('cat', categorical_transformer, categorical)])

    model = RandomForestClassifier(random_state=42)

    model = Pipeline(steps=[('preprocessor', transformations),
                        ('classifier', model)])

    model.fit(x_train, y_train)

    import joblib
    os.makedirs(f"{ugce_dir}/results/models", exist_ok=True)
    joblib.dump(model, f"{ugce_dir}/results/models/{datasetName}_model.pkl")

y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

negative_instances = x_test[model.predict(x_test) == 0]
instances_to_explain = negative_instances
print("Number of instances to explain: ", len(instances_to_explain))

Accuracy:  0.6320907617504052
Number of instances to explain:  550


In [6]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

In [7]:
numerical_columns = iea.dataset.select_dtypes(include=['int64', 'float64']).columns

non_zero_descriptions = {}

for col in numerical_columns:
    non_zero_values = iea.dataset[iea.dataset[col] != 0][col]
    if not non_zero_values.empty:
        non_zero_descriptions[col] = non_zero_values.describe()

# Display results
for feature, stats in non_zero_descriptions.items():
    print(f"\n Feature: {feature}")
    print(stats)


 Feature: sex
count    4994.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: sex, dtype: float64

 Feature: age
count    6167.000000
mean       34.531863
std        11.726167
min        18.000000
25%        25.000000
50%        31.000000
75%        42.000000
max        96.000000
Name: age, dtype: float64

 Feature: race
count    2994.000000
mean        2.510688
std         0.982669
min         1.000000
25%         2.000000
50%         2.000000
75%         3.000000
max         5.000000
Name: race, dtype: float64

 Feature: juv_fel_count
count    207.000000
mean       1.763285
std        1.847855
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       20.000000
Name: juv_fel_count, dtype: float64

 Feature: juv_misd_count
count    352.000000
mean       1.599432
std        1.392409
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       13.000000
Name

# UGCE

## Dynamic

# Only Immutability

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    'age': 'i',
}
for col in iea.feature_names:
    if col not in updated_constraints:
        updated_constraints[col] = ''
updated_constraints

results_incremental_explainer_immutability = []
for i in range(5):
    import time
    strategy = "fix_population_update_fitness"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="fix_population_update_fitness", population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer_immutability.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer_immutability, open(f'{results_dir}/results_incremental{strategy}_only_immutability_consts_ONE_constraint.pkl', 'wb'))

In [10]:
from test_utils import *

aggregate_results_incremental(iea, results_incremental_explainer_immutability, verbose=True)

Full Time: mean = 24.79, std = 0.00
Generations: mean = 9.95, std = 0.00
Coverage: mean = 33.70, std = 0.00
Proximity Loss: mean = 0.04, std = 0.00
Sparsity: mean = 0.03, std = 0.00
Intermediate Best Distances: mean = 0.04, std = 0.00


(24.785731077194214,
 9.946236559139784,
 33.69565217391305,
 0.039761270558577026,
 0.030997983870967742,
 0.036193807155807715)

# Only Range

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    'age': (32, 96)
}
for col in iea.feature_names:
    if col not in updated_constraints:
        updated_constraints[col] = ''
updated_constraints


results_incremental_explainer_ranges = []
for i in range(5):
    import time
    strategy = "fix_population_update_fitness"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="fix_population_update_fitness", population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer_ranges.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer_ranges, open(f'{results_dir}/results_incremental{strategy}_only_range_consts_ONE_constraint.pkl', 'wb'))

In [12]:
from test_utils import *

aggregate_results_incremental(iea, results_incremental_explainer_ranges, verbose=True)

Time taken for generating counterfactuals using UGCE dynamic:  1.3310338775316874  minutes
Full Time: mean = 1.33, std = 0.00
Generations: mean = 16.38, std = 0.00
Coverage: mean = 61.05, std = 0.00
Proximity Loss: mean = 0.06, std = 0.00
Sparsity: mean = 0.03, std = 0.00
Intermediate Best Distances: mean = 0.06, std = 0.00


(1.3310338775316874,
 16.379821958456972,
 61.050724637681164,
 0.05917771703397556,
 0.03389280415430267,
 0.057514404212170446)

# Only Directionality

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    "age": 'incr',
}
for col in iea.feature_names:
    if col not in updated_constraints:
        updated_constraints[col] = ''
updated_constraints


results_incremental_explainer_direct = []
for i in range(5):
    import time
    strategy = "fix_population_update_fitness"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="fix_population_update_fitness", population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer_direct.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer_direct, open(f'{results_dir}/results_incremental{strategy}_only_directionality_consts_ONE_constraint.pkl', 'wb'))

In [14]:
from test_utils import *

aggregate_results_incremental(iea, results_incremental_explainer_direct, verbose=True)

Full Time: mean = 12.56, std = 0.00
Generations: mean = 11.35, std = 0.00
Coverage: mean = 15.94, std = 0.00
Proximity Loss: mean = 0.05, std = 0.00
Sparsity: mean = 0.04, std = 0.00
Intermediate Best Distances: mean = 0.05, std = 0.00


(12.559275150299072,
 11.352272727272727,
 15.942028985507244,
 0.05117227004145386,
 0.03586647727272727,
 0.049164450051177734)